# Lineage h5ad Structure Inspector
Quick inspection of each lineage file to identify the correct label column.

## Cell 1 — Imports + File Paths

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sparse
from pathlib import Path

# ===== EDIT PATHS HERE =====
LINEAGE_FILES = {
    "epithelial": Path("/home/h2048/data/py/0122/epithelial_subcluster_v4_5_2_production/epithelial_with_subclusters_v4_5_2.h5ad"),
    "tcell"     : Path("/home/h2048/data/py/0318/tnk_subcluster_retrain/adata_tnk_scanvi_ref_retrain_v1_2.h5ad"),
    "myeloid"   : Path("/home/h2048/data/py/0209/myeloid_validation_optimized/adata_myeloid_refined_FINAL.h5ad"),
    "bcell"     : Path("/home/h2048/data/core20260115/adata_bcell_FINAL_corrected_20260114.h5ad"),
    "stromal"   : Path("/home/h2048/data/py/0308/stromal_reintegration_v1_3/stromal_reintegrated_scvi_scanvi_v1_3.h5ad"),
}

# Keywords used to flag annotation-like columns for inspection
ANNOTATION_KEYWORDS = [
    "cell_type", "celltype", "scanvi", "leiden", "cluster",
    "ann_", "label", "annotation", "lineage", "subtype",
    "refined", "pred", "majority", "phase", "Cell_Type",
]

print("[OK] paths configured")
for k, p in LINEAGE_FILES.items():
    exists = "OK" if p.exists() else "MISSING"
    print(f"  [{exists}] {k}: {p.name}")

## Cell 2 — Inspector Function

In [ ]:
def inspect_h5ad(name, path, top_n_cats=12, show_all_ann=True):
    """
    Load h5ad and print:
      - shape, .raw info, layer names, obsm keys
      - all annotation-like obs columns with value counts (top N)
      - raw type check for .raw.X and layers['counts']
    """
    print("=" * 72)
    print(f"  {name.upper()}  |  {path.name}")
    print("=" * 72)

    adata = sc.read_h5ad(path)

    # ── Basic shape ──────────────────────────────────────────────────────
    print(f"\nShape   : {adata.n_obs:,} cells x {adata.n_vars:,} genes")
    if adata.raw is not None:
        raw_X   = adata.raw.X
        raw_d   = raw_X.data if sparse.issparse(raw_X) else np.asarray(raw_X).ravel()
        _s      = raw_d[:min(50000, len(raw_d))]
        frac    = np.abs(_s - np.round(_s)).max() if len(_s) > 0 else 0
        raw_int = "integer-like" if frac < 1e-3 else f"NON-INTEGER (max_frac={frac:.4f})"
        print(f".raw    : {adata.raw.n_vars:,} genes  |  dtype={raw_X.dtype}  |  values={raw_int}")
    else:
        print(".raw    : None")

    if adata.layers:
        for lname, lX in adata.layers.items():
            ld   = lX.data if sparse.issparse(lX) else np.asarray(lX).ravel()
            _s   = ld[:min(50000, len(ld))]
            frac = np.abs(_s - np.round(_s)).max() if len(_s) > 0 else 0
            tag  = "integer-like" if frac < 1e-3 else f"non-integer(frac={frac:.3f})"
            fmt  = type(lX).__name__ if sparse.issparse(lX) else "ndarray"
            print(f"layer   : [{lname}]  {fmt}  dtype={lX.dtype}  {tag}")
    else:
        print("layers  : (none)")

    print(f"obsm    : {list(adata.obsm.keys())}")

    # ── Annotation columns ───────────────────────────────────────────────
    ann_cols = [
        col for col in adata.obs.columns
        if any(kw in col.lower() for kw in ANNOTATION_KEYWORDS)
        and col not in ("_scvi_batch","_scvi_labels",
                        "_scvi_extra_categorical_covs","_scvi_extra_continuous_covs")
    ]

    if show_all_ann:
        print(f"\n{'─'*72}")
        print(f"Annotation-like obs columns ({len(ann_cols)} found):")
        print(f"{'─'*72}")
        for col in ann_cols:
            series = adata.obs[col]
            dtype  = str(series.dtype)
            n_uniq = series.nunique()
            n_null = int(series.isna().sum())
            # top value counts
            try:
                vc = series.value_counts(dropna=False).head(top_n_cats)
                vc_str = "  |  ".join(f"{v}({n})" for v, n in vc.items())
            except Exception:
                vc_str = "(cannot compute)"
            null_tag = f"  [{n_null} null]" if n_null > 0 else ""
            print(f"\n  [{col}]  dtype={dtype}  unique={n_uniq}{null_tag}")
            print(f"  {vc_str}")

    # ── Non-annotation obs columns summary ───────────────────────────────
    other_cols = [c for c in adata.obs.columns if c not in ann_cols]
    print(f"\n{'─'*72}")
    print(f"Other obs columns ({len(other_cols)}): {other_cols}")

    del adata
    import gc; gc.collect()
    print()

print("[OK] inspect_h5ad defined")

## Epithelial

In [ ]:
inspect_h5ad("epithelial", LINEAGE_FILES["epithelial"])

## Tcell

In [ ]:
inspect_h5ad("tcell", LINEAGE_FILES["tcell"])

## Myeloid

In [ ]:
inspect_h5ad("myeloid", LINEAGE_FILES["myeloid"])

## Bcell

In [ ]:
inspect_h5ad("bcell", LINEAGE_FILES["bcell"])

## Stromal

In [ ]:
inspect_h5ad("stromal", LINEAGE_FILES["stromal"])

## Summary — Candidate Label Columns

In [ ]:
print("="*72)
print("CANDIDATE LABEL COLUMN SUMMARY")
print("="*72)
print("Fill in the 'chosen' column based on inspection above.")
print()

rows = []
for name, path in LINEAGE_FILES.items():
    adata = sc.read_h5ad(path)
    ann_cols = [
        col for col in adata.obs.columns
        if any(kw in col.lower() for kw in ANNOTATION_KEYWORDS)
        and col not in ("_scvi_batch","_scvi_labels",
                        "_scvi_extra_categorical_covs","_scvi_extra_continuous_covs")
    ]
    for col in ann_cols:
        n_uniq = adata.obs[col].nunique()
        n_null = int(adata.obs[col].isna().sum())
        rows.append({"lineage": name, "column": col,
                     "unique": n_uniq, "null": n_null})
    del adata
    import gc; gc.collect()

df = pd.DataFrame(rows)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)
print(df.to_string(index=False))